# CFPB Credit Card Complaints — January 2025

## Goal
Prepare and validate credit card complaint data for a Databricks dashboard using Bronze, Silver, and Gold tables.

This is an independent portfolio project, not a client engagement.

## Data and scope
- Source: CFPB Consumer Complaint Database.
- Input file: `cfpb_credit_card_2025_01_raw.csv`.
- Scope: complaints received January 1–31, 2025, for the Credit card product.
- The original run parsed 9,073 records across 16 source columns.
- The downloaded count has not been reconciled with the source website's result count.

## Environment and prerequisites
This notebook is designed for Databricks with PySpark, Spark SQL, Delta tables, and Unity Catalog volumes. It does not run unchanged in a standard local Jupyter environment.

Before running:
- Provide the original CSV in a volume accessible to your account.
- Update the input paths and table names for your environment.
- Ensure the target catalog and schema exist and you have the required permissions.
- Use a separate development schema with unused target table names.

## Execution safety
The table-creation steps use `.mode("error")` or `CREATE TABLE`.
They intentionally stop if a target table already exists.

Do not run all cells against the existing project tables.
Do not drop or overwrite existing tables just to rerun this notebook.

For a first run in a prepared development environment, execute cells in order and review each validation result before continuing.

## Public version
Saved outputs have been cleared, and the individual complaint preview has been removed.
The original notebook retains the historical validation outputs.

This edited public version has not been rerun end-to-end.
The counts and findings in the validation notes describe the original run.

## Interpretation
- Forwarding interval is not company response time.
- Timely response does not measure customer satisfaction or resolution quality.
- Complaint counts do not represent all credit card customers.

In [0]:
# Lokasi folder dataset di Databricks
volume_path = "/Volumes/workspace/cfpb/raw_files"

# Tampilkan daftar file dalam folder
display(dbutils.fs.ls(volume_path))

In [0]:
# Lokasi lengkap file CSV
file_path = (
    "/Volumes/workspace/cfpb/raw_files/"
    "cfpb_credit_card_2025_01_raw.csv"
)

# Baca semua kolom sebagai teks terlebih dahulu
df_raw = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("inferSchema", "false")
    .option("mode", "FAILFAST")
    .csv(file_path)
)

# Periksa struktur dan jumlah record yang terbaca
print("Jumlah kolom:", len(df_raw.columns))
print("Jumlah record:", df_raw.count())
print("\nNama kolom:")
for column_name in df_raw.columns:
    print("-", column_name)

In [0]:
# Hitung jumlah record untuk setiap produk
display(
    df_raw
    .groupBy("Product")
    .count()
    .orderBy("count", ascending=False)
)

In [0]:
from pyspark.sql import functions as F

# Buat kolom pemeriksaan tanpa mengubah df_raw
df_check = (
    df_raw
    .withColumn(
        "received_date_check",
        F.expr("try_cast(substring(`Date received`, 1, 10) AS DATE)")
    )
    .withColumn(
        "complaint_id_check",
        F.trim(F.col("Complaint ID"))
    )
)

# Ringkasan kualitas awal
summary = df_check.agg(
    F.count("*").alias("total_records"),

    F.min("received_date_check").alias("earliest_date"),
    F.max("received_date_check").alias("latest_date"),

    F.count(
        F.when(F.col("received_date_check").isNull(), 1)
    ).alias("missing_or_invalid_dates"),

    F.count(
        F.when(
            (F.col("received_date_check") < F.lit("2025-01-01").cast("date")) |
            (F.col("received_date_check") > F.lit("2025-01-31").cast("date")),
            1
        )
    ).alias("records_outside_january"),

    F.count(
        F.when(
            F.col("complaint_id_check").isNull() |
            (F.col("complaint_id_check") == ""),
            1
        )
    ).alias("missing_or_blank_ids"),

    F.countDistinct(
        F.when(
            F.col("complaint_id_check") != "",
            F.col("complaint_id_check")
        )
    ).alias("unique_nonblank_ids")
)

display(summary)

In [0]:
# Tampilkan hasil pemeriksaan ID agar tidak terpotong
summary.select(
    "missing_or_blank_ids",
    "unique_nonblank_ids"
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F

# Kolom yang akan digunakan dalam analisis
columns_to_check = [
    "Company",
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "State",
    "Submitted via",
    "Date sent to company",
    "Company response to consumer",
    "Timely response?"
]

# Hitung nilai null, teks kosong, atau hanya berisi spasi
missing_counts = df_raw.agg(*[
    F.count(
        F.when(
            F.col(column).isNull() |
            (F.trim(F.col(column)) == ""),
            1
        )
    ).alias(column)
    for column in columns_to_check
])

# Tampilkan secara vertikal supaya mudah dibaca
missing_counts.show(vertical=True, truncate=False)

In [0]:
# Lihat nilai asli dan jumlahnya tanpa mengganti kategori
display(
    df_raw
    .groupBy("Timely response?")
    .count()
    .orderBy("count", ascending=False)
)

In [0]:
from pyspark.sql import functions as F

# Tambahkan tanggal pengiriman untuk pemeriksaan
df_dates = (
    df_check
    .withColumn(
        "sent_date_check",
        F.expr(
            "try_cast(substring(`Date sent to company`, 1, 10) AS DATE)"
        )
    )
    .withColumn(
        "forwarding_days",
        F.datediff(
            F.col("sent_date_check"),
            F.col("received_date_check")
        )
    )
)

# Periksa validitas dan urutan tanggal
date_summary = df_dates.agg(
    F.count("*").alias("total_records"),

    F.count(
        F.when(F.col("sent_date_check").isNull(), 1)
    ).alias("invalid_sent_dates"),

    F.count(
        F.when(F.col("forwarding_days") < 0, 1)
    ).alias("sent_before_received"),

    F.min("forwarding_days").alias("min_forwarding_days"),
    F.max("forwarding_days").alias("max_forwarding_days"),

    F.round(
        F.avg("forwarding_days"), 2
    ).alias("avg_forwarding_days")
)

date_summary.show(vertical=True, truncate=False)

In [0]:
# Public version: individual complaint details are not displayed.
# The original notebook retains the row-level inspection.
# Aggregate forwarding-interval checks remain in the preceding cell.
# Forwarding interval is not company response time.

In [0]:
# Tampilkan kategori respons perusahaan dan jumlah record
(
    df_raw
    .groupBy("Company response to consumer")
    .count()
    .orderBy("count", ascending=False)
    .show(truncate=False)
)

In [0]:
# Bandingkan kategori respons dengan indikator ketepatan waktu
(
    df_raw
    .groupBy(
        "Company response to consumer",
        "Timely response?"
    )
    .count()
    .orderBy(
        "Company response to consumer",
        "Timely response?"
    )
    .show(truncate=False)
)

## Initial Data Validation — January 2025

### Dataset scope
- Source: CFPB Consumer Complaint Database.
- File: cfpb_credit_card_2025_01_raw.csv.
- Parsed records: 9,073 across 16 columns.
- Product: Credit card.
- Date received range: January 1–31, 2025.
- The downloaded record count has not yet been reconciled
  with the result count displayed on the source website.

### Checks completed
- No missing or blank complaint IDs.
- All 9,073 complaint IDs are unique after trimming whitespace.
- No missing or invalid received dates.
- No received dates outside January 2025.
- No null or blank values in the 10 selected analytical columns.
- No invalid sent dates or sent dates preceding received dates.

### Findings requiring careful interpretation
- Forwarding interval: 0–97 calendar days; mean 0.72 days.
- The 97-day interval agrees with the displayed source dates,
  but its cause has not been established.
- Timely response: Yes = 9,047; No = 26.
- Of the 26 records marked No:
  - 17 are Closed with explanation.
  - 1 is Closed with non-monetary relief.
  - 8 are Untimely response.

### Processing decisions
- Retain all records at this stage.
- Preserve response category and timeliness as separate fields.
- Do not treat forwarding days as company response time.
- Do not interpret closure as consumer satisfaction.
- Exclude free-text consumer narratives from the first analytical table.
- Preserve the original CSV unchanged.

### Status
Initial checks passed within the tested scope.
This does not establish complete data accuracy or representativeness.

In [0]:
import re
from pyspark.sql import functions as F

# Rapikan nama kolom:
# "Date received" -> "date_received"
# "Timely response?" -> "timely_response"
bronze_columns = [
    re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    for name in df_raw.columns
]

# Pastikan tidak ada nama kolom yang menjadi sama
assert len(bronze_columns) == len(set(bronze_columns))

# Pertahankan seluruh nilai sumber, lalu tambahkan metadata
df_bronze = (
    df_raw.toDF(*bronze_columns)
    .withColumn("_source_file", F.lit(file_path))
    .withColumn("_ingested_at", F.current_timestamp())
)

df_bronze.printSchema()

In [0]:
bronze_table = "workspace.cfpb.bronze_complaints"

(
    df_bronze.write
    .format("delta")
    .mode("error")
    .saveAsTable(bronze_table)
)

print(f"Tabel berhasil dibuat: {bronze_table}")

In [0]:
bronze_saved = spark.table("workspace.cfpb.bronze_complaints")

bronze_saved.agg(
    F.count("*").alias("total_records"),
    F.countDistinct(
        F.trim(F.col("complaint_id"))
    ).alias("unique_ids"),
    F.countDistinct("_source_file").alias("source_files")
).show(truncate=False)

print("Jumlah kolom:", len(bronze_saved.columns))

In [0]:
from pyspark.sql import functions as F

# Ambil data dari tabel yang sudah tersimpan
bronze_source = spark.table("workspace.cfpb.bronze_complaints")

print("Jumlah record Bronze:", bronze_source.count())

In [0]:
# Narasi tetap tersedia di Bronze, tetapi tidak masuk Silver awal
silver_text = bronze_source.drop("consumer_complaint_narrative")

# Pilih kolom teks sumber, tanpa mengubah metadata
text_columns = [
    name
    for name, dtype in silver_text.dtypes
    if dtype == "string" and not name.startswith("_")
]

# Hilangkan spasi di awal/akhir.
# Teks kosong menjadi null; nilai lainnya tetap dipertahankan.
for name in text_columns:
    cleaned_value = F.trim(F.col(name))

    silver_text = silver_text.withColumn(
        name,
        F.when(cleaned_value == "", F.lit(None))
        .otherwise(cleaned_value)
    )

print("Jumlah kolom setelah narasi dikeluarkan:", len(silver_text.columns))

In [0]:
df_silver = (
    silver_text
    .withColumn(
        "date_received",
        F.expr("try_cast(substring(date_received, 1, 10) AS DATE)")
    )
    .withColumn(
        "date_sent_to_company",
        F.expr(
            "try_cast(substring(date_sent_to_company, 1, 10) AS DATE)"
        )
    )
    .withColumn(
        "forwarding_days",
        F.datediff("date_sent_to_company", "date_received")
    )
)

df_silver.printSchema()

In [0]:
silver_checks = df_silver.agg(
    F.count("*").alias("total_records"),

    F.countDistinct("complaint_id").alias("unique_ids"),

    F.count(
        F.when(F.col("complaint_id").isNull(), 1)
    ).alias("missing_ids"),

    F.count(
        F.when(F.col("date_received").isNull(), 1)
    ).alias("missing_or_invalid_received_dates"),

    F.count(
        F.when(F.col("date_sent_to_company").isNull(), 1)
    ).alias("missing_or_invalid_sent_dates"),

    F.count(
        F.when(F.col("forwarding_days") < 0, 1)
    ).alias("negative_forwarding_days"),

    F.min("forwarding_days").alias("min_forwarding_days"),
    F.max("forwarding_days").alias("max_forwarding_days"),
    F.round(
        F.avg("forwarding_days"), 2
    ).alias("avg_forwarding_days")
)

silver_checks.show(vertical=True, truncate=False)

In [0]:
silver_table = "workspace.cfpb.silver_complaints"

(
    df_silver.write
    .format("delta")
    .mode("error")
    .saveAsTable(silver_table)
)

print(f"Tabel berhasil dibuat: {silver_table}")

In [0]:
silver_saved = spark.table("workspace.cfpb.silver_complaints")

print("Jumlah record tersimpan:", silver_saved.count())
print("Jumlah kolom:", len(silver_saved.columns))

# Samakan urutan kolom sebelum membandingkan
saved_aligned = silver_saved.select(*df_silver.columns)

# Bandingkan seluruh nilai, termasuk jumlah kemunculan baris
missing_rows = df_silver.exceptAll(saved_aligned).count()
extra_rows = saved_aligned.exceptAll(df_silver).count()

print("Baris hasil transformasi yang tidak tersimpan:", missing_rows)
print("Baris tambahan atau berbeda di tabel:", extra_rows)

assert missing_rows == 0 and extra_rows == 0, (
    "Isi tabel berbeda dari hasil transformasi. Jangan lanjut dahulu."
)

print("LULUS: isi tabel sama dengan hasil transformasi Silver.")

In [0]:
%sql
WITH monthly_counts AS (
    SELECT
        TRUNC(date_received, 'MONTH') AS received_month,
        product,
        COUNT(*) AS total_complaints,
        COUNT_IF(timely_response = 'Yes') AS timely_yes,
        COUNT_IF(timely_response = 'No') AS timely_no,
        COUNT_IF(
            timely_response IS NULL
            OR timely_response NOT IN ('Yes', 'No')
        ) AS timely_unknown
    FROM workspace.cfpb.silver_complaints
    WHERE date_received >= DATE '2025-01-01'
      AND date_received < DATE '2025-02-01'
      AND product = 'Credit card'
    GROUP BY
        TRUNC(date_received, 'MONTH'),
        product
)
SELECT
    received_month,
    product,
    total_complaints,
    timely_yes,
    timely_no,
    timely_unknown,
    ROUND(
        100.0 * timely_yes
        / NULLIF(timely_yes + timely_no, 0),
        2
    ) AS timely_response_pct
FROM monthly_counts
ORDER BY received_month, product;

In [0]:
%sql
SELECT
    TRUNC(date_received, 'MONTH') AS received_month,
    product,
    company_response_to_consumer AS response_category,
    COUNT(*) AS total_complaints,
    COUNT_IF(timely_response = 'Yes') AS timely_yes,
    COUNT_IF(timely_response = 'No') AS timely_no,
    COUNT_IF(
        timely_response IS NULL
        OR timely_response NOT IN ('Yes', 'No')
    ) AS timely_unknown
FROM workspace.cfpb.silver_complaints
WHERE date_received >= DATE '2025-01-01'
  AND date_received < DATE '2025-02-01'
  AND product = 'Credit card'
GROUP BY
    TRUNC(date_received, 'MONTH'),
    product,
    company_response_to_consumer
ORDER BY total_complaints DESC, response_category;

In [0]:
%sql
CREATE TABLE workspace.cfpb.gold_monthly_summary
USING DELTA
AS
WITH monthly_counts AS (
    SELECT
        TRUNC(date_received, 'MONTH') AS received_month,
        product,
        COUNT(*) AS total_complaints,
        COUNT_IF(timely_response = 'Yes') AS timely_yes,
        COUNT_IF(timely_response = 'No') AS timely_no,
        COUNT_IF(
            timely_response IS NULL
            OR timely_response NOT IN ('Yes', 'No')
        ) AS timely_unknown
    FROM workspace.cfpb.silver_complaints
    WHERE date_received >= DATE '2025-01-01'
      AND date_received < DATE '2025-02-01'
      AND product = 'Credit card'
    GROUP BY
        TRUNC(date_received, 'MONTH'),
        product
)
SELECT
    received_month,
    product,
    total_complaints,
    timely_yes,
    timely_no,
    timely_unknown,
    ROUND(
        100.0 * timely_yes
        / NULLIF(timely_yes + timely_no, 0),
        2
    ) AS timely_response_pct
FROM monthly_counts
ORDER BY received_month, product;

In [0]:
%sql
CREATE TABLE workspace.cfpb.gold_monthly_response
USING DELTA
AS
SELECT
    TRUNC(date_received, 'MONTH') AS received_month,
    product,
    company_response_to_consumer AS response_category,
    COUNT(*) AS total_complaints,
    COUNT_IF(timely_response = 'Yes') AS timely_yes,
    COUNT_IF(timely_response = 'No') AS timely_no,
    COUNT_IF(
        timely_response IS NULL
        OR timely_response NOT IN ('Yes', 'No')
    ) AS timely_unknown
FROM workspace.cfpb.silver_complaints
WHERE date_received >= DATE '2025-01-01'
  AND date_received < DATE '2025-02-01'
  AND product = 'Credit card'
GROUP BY
    TRUNC(date_received, 'MONTH'),
    product,
    company_response_to_consumer
ORDER BY total_complaints DESC, response_category;

In [0]:
%sql
SELECT
    'gold_monthly_summary' AS table_name,
    COUNT(*) AS stored_rows,
    SUM(total_complaints) AS total_complaints,
    SUM(timely_yes) AS timely_yes,
    SUM(timely_no) AS timely_no,
    SUM(timely_unknown) AS timely_unknown
FROM workspace.cfpb.gold_monthly_summary

UNION ALL

SELECT
    'gold_monthly_response' AS table_name,
    COUNT(*) AS stored_rows,
    SUM(total_complaints) AS total_complaints,
    SUM(timely_yes) AS timely_yes,
    SUM(timely_no) AS timely_no,
    SUM(timely_unknown) AS timely_unknown
FROM workspace.cfpb.gold_monthly_response;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW daily_summary_preview AS
SELECT
    date_received AS received_date,
    product,
    COUNT(*) AS total_complaints,
    COUNT_IF(timely_response = 'Yes') AS timely_yes,
    COUNT_IF(timely_response = 'No') AS timely_no,
    COUNT_IF(
        timely_response IS NULL
        OR timely_response NOT IN ('Yes', 'No')
    ) AS timely_unknown
FROM workspace.cfpb.silver_complaints
WHERE date_received >= DATE '2025-01-01'
  AND date_received < DATE '2025-02-01'
  AND product = 'Credit card'
GROUP BY date_received, product;

In [0]:
%sql
SELECT *
FROM daily_summary_preview
ORDER BY received_date, product;

In [0]:
%sql
SELECT
    COUNT(*) AS daily_rows,
    COUNT(DISTINCT received_date) AS dates_with_records,
    MIN(received_date) AS first_date,
    MAX(received_date) AS last_date,
    SUM(total_complaints) AS total_complaints,
    SUM(timely_yes) AS timely_yes,
    SUM(timely_no) AS timely_no,
    SUM(timely_unknown) AS timely_unknown,
    COUNT_IF(
        total_complaints <> timely_yes + timely_no + timely_unknown
    ) AS inconsistent_daily_rows
FROM daily_summary_preview;

In [0]:
%sql
CREATE TABLE workspace.cfpb.gold_daily_summary
USING DELTA
AS
SELECT
    received_date,
    product,
    total_complaints,
    timely_yes,
    timely_no,
    timely_unknown
FROM daily_summary_preview;

In [0]:
%sql
SELECT
    COUNT(*) AS stored_rows,
    COUNT(DISTINCT received_date) AS dates_with_records,
    MIN(received_date) AS first_date,
    MAX(received_date) AS last_date,
    SUM(total_complaints) AS total_complaints,
    SUM(timely_yes) AS timely_yes,
    SUM(timely_no) AS timely_no,
    SUM(timely_unknown) AS timely_unknown,
    COUNT_IF(
        total_complaints <> timely_yes + timely_no + timely_unknown
    ) AS inconsistent_daily_rows
FROM workspace.cfpb.gold_daily_summary;